In [7]:
# Install dependencies INTO THE RUNNING KERNEL.
# Use %pip (not !pip): %pip targets this notebook's interpreter, so packages land
# in the same Python that runs the imports below. `!pip` shells out to whatever pip
# is first on PATH (here: anaconda's python 3.12) — a *different* environment than
# this kernel (python.org 3.13) — which is why a prior run reported "Requirement
# already satisfied" yet the next cell raised ModuleNotFoundError.
#
# - azure-identity: ActiveDirectoryInteractive (browser auth in the ODBC connection
#   string) is Windows-only and never works on macOS. We instead fetch an AAD token
#   in Python (azure-identity opens the browser / handles MFA) and hand it to pyodbc.
# - sqlalchemy: required by pandas .to_sql() to write df_full_merged back to SQL.
# - python-dotenv: loads the git-ignored .env file that holds the connection config
#   (server / database / tenant id) so no secrets live in this notebook.
%pip install -q pyodbc azure-identity sqlalchemy python-dotenv

import sys, pyodbc
print(f"kernel python : {sys.executable}")
print(f"pyodbc        : {pyodbc.version}")
print(f"ODBC drivers  : {pyodbc.drivers()}")  # must list 'ODBC Driver 18 for SQL Server'

Note: you may need to restart the kernel to use updated packages.
kernel python : /usr/local/bin/python3
pyodbc        : 5.3.0
ODBC drivers  : ['ODBC Driver 18 for SQL Server']


In [8]:
import os
import struct
import pyodbc
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv, find_dotenv

# Why this approach (and not Authentication=ActiveDirectoryInteractive):
# the ODBC driver's built-in interactive/browser auth is Windows-only — on macOS
# it never launches a browser and just times out. Cross-platform fix: get the AAD
# access token in Python with azure-identity (opens the browser, handles MFA), then
# pass the raw token to pyodbc via the SQL_COPT_SS_ACCESS_TOKEN pre-connect attribute.

# Connection config is read from a git-ignored .env at the repo root (never committed).
# find_dotenv() walks up from this notebook's folder to locate it. Copy .env.example
# -> .env and fill in your values before running this cell.
load_dotenv(find_dotenv())

server   = os.environ["AZURE_SQL_SERVER"]     # e.g. <name>.database.windows.net
database = os.environ["AZURE_SQL_DATABASE"]

# This Gmail is a personal Microsoft account; its *home* tenant is the consumer
# "Microsoft Services" tenant, NOT the directory that owns this database. When you
# created an Azure subscription with this Gmail, Azure auto-created a "Default
# Directory" (an Entra ID org tenant) and added you to it — that is where the DB
# lives. Pin sign-in to that tenant, or it defaults to the consumer tenant and fails
# with "account does not exist in tenant 'Microsoft Services'".
TENANT_ID = os.environ["AZURE_TENANT_ID"]     # your default directory's tenant id

# Connection string carries NO UID / Authentication — auth comes from the token below.
conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=60;"
)

SQL_COPT_SS_ACCESS_TOKEN = 1256              # pyodbc attr id for an AAD access token
AAD_SCOPE = "https://database.windows.net/.default"

try:
    print("Opening a browser to sign in to Azure AD — complete the login there...")
    # tenant_id pins sign-in to the org tenant that owns the DB.
    credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)
    token = credential.get_token(AAD_SCOPE).token

    # ODBC expects the token as UTF-16-LE bytes prefixed with a 4-byte length
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})
    print("✅ Successfully connected to the Azure SQL Database!")

    cursor = conn.cursor()
    cursor.execute("SELECT @@VERSION")
    print(f"Server version information:\n{cursor.fetchone()[0]}")
    cursor.close()
    conn.close()

except Exception as e:
    print(f"❌ Connection failed. Error message:\n{e}")
    print(
        "\nTroubleshooting:\n"
        "  • 'account does not exist in tenant Microsoft Services' → TENANT_ID is\n"
        "    wrong; set it to the org tenant's ID (Entra ID → Overview).\n"
        "  • 'Login failed for user <token-identified principal>' → token is fine but\n"
        "    this account has no DB access yet; needs CREATE USER ... FROM EXTERNAL\n"
        "    PROVIDER + a role (run by a DB admin).\n"
        "  • 'Client with IP address ... is not allowed' → add your IP to the Azure\n"
        "    SQL server firewall in the Azure portal."
    )

Opening a browser to sign in to Azure AD — complete the login there...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/msal/oauth2cli/oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


✅ Successfully connected to the Azure SQL Database!
Server version information:
Microsoft SQL Azure (RTM) - 12.0.2000.8 
	Apr 14 2026 20:27:12 
	Copyright (C) 2025 Microsoft Corporation



In [9]:
# === Upload df_full_merged to Azure SQL ===
# Depends on the connection cell above: it reuses TENANT_ID / server / database and the
# `credential` object created there. Reusing that same credential means its in-memory
# token cache is already populated, so get_token() returns silently — no second browser
# popup in the same session.
import struct
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import NVARCHAR

DATA_PATH = '../data/processed/df_full_merged.csv'
# ====================================================
# parse_dates so these land as SQL datetime, not NVARCHAR text
df_full_merged = pd.read_csv(
    DATA_PATH,
    parse_dates=['date_of_birth', 'intake_date', 'outcome_date'],
)

TABLE_NAME = "df_full_merged"

# --- AAD token -> ODBC access-token struct (reuse the connection cell's credential) ---
SQL_COPT_SS_ACCESS_TOKEN = 1256
token = credential.get_token(AAD_SCOPE).token   # served silently from the cached credential
token_bytes = token.encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

odbc_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60;"
)
# fast_executemany=True batches the INSERTs — essential for 160k rows.
engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(odbc_str),
    connect_args={"attrs_before": {SQL_COPT_SS_ACCESS_TOKEN: token_struct}},
    fast_executemany=True,
)

# Right-size NVARCHAR per string column (pandas would otherwise default every text
# column to NVARCHAR(max), which can't be indexed and is slower to write).
dtype_map = {}
for col in df_full_merged.columns:
    s = df_full_merged[col]
    if s.dtype == "object" or pd.api.types.is_string_dtype(s):
        maxlen = int(s.astype("string").str.len().max() or 1)
        dtype_map[col] = NVARCHAR(length=None) if maxlen > 2000 \
            else NVARCHAR(length=min(max(maxlen + 16, 32), 4000))

print(f"Uploading {len(df_full_merged):,} rows × {df_full_merged.shape[1]} cols "
      f"to [{TABLE_NAME}] (if_exists='replace')...")
df_full_merged.to_sql(
    TABLE_NAME, engine,
    if_exists="replace",   # drop & recreate; use 'append' to add to an existing table
    index=False,
    chunksize=1000,
    dtype=dtype_map,
)

# Verify the row count landed in SQL
with engine.connect() as cx:
    n = cx.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
print(f"✅ Done. [{TABLE_NAME}] now holds {n:,} rows in SQL.")

Uploading 162,465 rows × 26 cols to [df_full_merged] (if_exists='replace')...
✅ Done. [df_full_merged] now holds 162,465 rows in SQL.


In [10]:
# === Verify the upload by querying SQL (reuses `engine` / TABLE_NAME from above) ===
import pandas as pd
from sqlalchemy import text

with engine.connect() as cx:
    # 1) row count in SQL vs the in-memory DataFrame
    n_sql = cx.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
    print(f"Rows in SQL:       {n_sql:,}")
    print(f"Rows in DataFrame: {len(df_full_merged):,}")
    print(f"Match: {'✅' if n_sql == len(df_full_merged) else '❌ MISMATCH'}")
    print("=" * 60)

    # 2) the table schema SQL actually created (column name / type / nullability)
    schema = pd.read_sql(text(
        "SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH, IS_NULLABLE "
        "FROM INFORMATION_SCHEMA.COLUMNS "
        "WHERE TABLE_NAME = :t ORDER BY ORDINAL_POSITION"
    ), cx, params={"t": TABLE_NAME})
    print("Column schema as stored in SQL:")
    display(schema)

    # 3) a few sample rows straight from the database
    print("\nSample rows from SQL:")
    sample = pd.read_sql(text(f"SELECT TOP 5 * FROM {TABLE_NAME}"), cx)
    display(sample)

    # 4) quick sanity aggregate — long-stay rate by animal_type, computed in SQL
    print("is_long_stay rate by animal_type (computed in SQL):")
    agg = pd.read_sql(text(
        f"SELECT animal_type, COUNT(*) AS n, AVG(CAST(is_long_stay AS float)) AS long_stay_rate "
        f"FROM {TABLE_NAME} GROUP BY animal_type ORDER BY n DESC"
    ), cx)
    display(agg)

Rows in SQL:       162,465
Rows in DataFrame: 162,465
Match: ✅
Column schema as stored in SQL:


,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
0,animal_id,nvarchar,32.0,YES
1,name,nvarchar,46.0,YES
2,animal_type,nvarchar,32.0,YES
3,sex,nvarchar,32.0,YES
4,primary_breed,nvarchar,50.0,YES
5,is_mix,bigint,NaN,YES
6,date_of_birth,datetime,NaN,YES
7,primary_color,nvarchar,32.0,YES
8,pattern,nvarchar,32.0,YES
9,secondary_color,nvarchar,32.0,YES



Sample rows from SQL:


,animal_id,name,animal_type,sex,primary_breed,is_mix,date_of_birth,primary_color,pattern,secondary_color,...,outcome_date,outcome_type,outcome_subtype,length_of_stay_days,is_long_stay,is_adopted,outcome_category,intake_year,intake_month,intake_weekday
0,A711373,NaN,Cat,Male,Domestic Shorthair,1,2015-08-22,Brown,Tabby,NaN,...,2015-09-06,Transfer,Partner,0,0,0,transfer,2015,9,6
1,A711187,Pappy,Dog,Male,Dachshund,1,2007-09-03,Tan,Solid,NaN,...,2015-09-06,Adoption,NaN,3,0,1,adopted,2015,9,3
2,A710890,The Dude,Dog,Male,Great Pyrenees,1,2012-08-29,White,Solid,NaN,...,2015-09-06,Adoption,NaN,8,0,1,adopted,2015,8,5
3,A529648,Bruno,Dog,Male,Miniature Pinscher,0,2005-12-07,Black,Solid,Tan,...,2015-09-06,Return to Owner,NaN,0,0,0,return to owner,2015,9,6
4,A711425,Oreo,Dog,Male,Shiba Inu,0,2015-04-06,Black,Solid,Tan,...,2015-09-06,Return to Owner,NaN,0,0,0,return to owner,2015,9,6


is_long_stay rate by animal_type (computed in SQL):


,animal_type,n,long_stay_rate
0,Dog,93728,0.156581
1,Cat,68737,0.253284
